In [1]:
# Parameters
PLOT_FOLDER = "/gpfs/home6/draju/A6/TabPFN/RF_TabPFN/SLURMGamma"
TEST_ROWS = None
SEED = 50005


In [2]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate

In [3]:
PLOT_FOLDER = "SLURMGamma"
target      = "gamma"
SEED        = 50005
TEST_ROWS   = None

In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn_extensions",
)

try:
    import tabpfn
    import tabpfn_extensions
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    from tabpfn_extensions.rf_pfn import RandomForestTabPFNRegressor

    os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

    print(f"TabPFN version: {tabpfn.__version__}")
    print("Selected model version:", ModelVersion.V2)
    print("Using RandomForestTabPFNRegressor")

except ImportError as exc:
    raise ImportError("tabpfn and tabpfn_extensions must be installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")


TabPFN version: 7.1.1
Selected model version: ModelVersion.V2
Using RandomForestTabPFNRegressor
Thread limit set to 16 (SLURM_CPUS_PER_TASK)


In [5]:
df = pd.read_csv("/gpfs/home6/draju/A6/DATASET_A4/interfacial_results_dataset_A4.csv")
print(f"Number of rows: {len(df)}")

if TEST_ROWS is not None:
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())

Number of rows: 19361
Full mode: using all rows
Total samples: 19361

gamma statistics:
count    19361.000000
mean         9.986390
std          6.163645
min          0.057838
25%          4.530057
50%          9.545818
75%         15.238060
max         22.536975
Name: gamma, dtype: float64


In [6]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   13552
Testing samples:    2904
Validation samples: 2905


In [7]:
# Train RF + TabPFN model
tabpfn_base = TabPFNRegressor(
    random_state=SEED,
    ignore_pretraining_limits=True,
    fit_mode="fit_preprocessors",
)
rf_tabpfn_model = RandomForestTabPFNRegressor(
    tabpfn=tabpfn_base,
    n_estimators=10,
    max_depth=3,
)
rf_tabpfn_model.fit(X_train, y_train)

y_train_pred = rf_tabpfn_model.predict(X_train)
y_test_pred  = rf_tabpfn_model.predict(X_test)
y_val_pred   = rf_tabpfn_model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="mN/m", y_val=y_val, y_val_pred=y_val_pred)


Model Performance for gamma

Training Set:
  R²:   0.999997
  RMSE: 0.010230 mN/m
  MAE:  0.007306 mN/m

Test Set:
  R²:   0.999993
  RMSE: 0.016146 mN/m
  MAE:  0.010312 mN/m

Validation Set:
  R²:   0.999995
  RMSE: 0.014358 mN/m
  MAE:  0.010000 mN/m


In [8]:
results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred,  y_test_pred,  y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
os.makedirs(PLOT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

model_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_model.joblib")
joblib.dump(rf_tabpfn_model, model_path)
print(f"Model saved to: {model_path}")

Predictions saved: 19361 rows


Model saved to: SLURMGamma/RF_TabPFN_gamma_model.joblib


In [9]:
cv_results = cross_validate(
    rf_tabpfn_model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=1,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")

Cross-Validation R² Scores:   [0.9997683  0.99863182 0.99945229 0.99935916 0.99929066]
Mean CV R²:   0.999300 (+/- 0.000744)

Cross-Validation RMSE Scores: [0.09497165 0.2293536  0.1441365  0.15489372 0.16221963]
Mean CV RMSE: 0.157115 (+/- 0.086161)

Cross-Validation MAE Scores:  [0.06594412 0.11790426 0.07924933 0.08674231 0.10632923]
Mean CV MAE:  0.091234 (+/- 0.037337)


In [10]:
metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
metrics["cv_r2_std"]      = float(cv_r2_scores.std())
metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
metrics["cv_mae_std"]     = float(cv_mae_scores.std())
metrics["model"]          = "RF_TabPFN"
metrics["features"]       = features
metrics["target"]         = target
metrics["seed"]           = SEED

os.makedirs(PLOT_FOLDER, exist_ok=True)
metrics_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nMetrics saved to: {metrics_path}")


Metrics saved to: SLURMGamma/RF_TabPFN_gamma_metrics.json


In [11]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 86.29 minutes
